### Analytical Evaluation of the Effective Dynamics (FAPT)

While the numerical propagation of the effective state is efficient for single pulse sequences, large parameter sweeps and pulse optimizations require a less computationally demanding approach. To achieve this, the perturbative construction can be evaluated analytically.

Instead of performing the time-evolution numerically over many small steps $\delta t$, this approach uses closed-form symbolic expressions for both the transformation matrix $W(\boldsymbol\lambda)$ and the effective Hamiltonian $H_{\mathrm{eff,ad}}$. The time evolution operator $U(t, t_i)$ over the entire pulse duration is then approximated using the **Magnus expansion** (up to second order), effectively evaluating the evolution in a single analytical step. 

The following code tests this analytical framework by substituting symbolic pulse shapes (like a Gaussian amplitude) into the analytically derived Magnus operator.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import sympy as sp
from utils import * 
from FAPT import * 
from pulse_optimization import * 
s = Simulation()
import matplotlib.pyplot as plt
rH = 2
rW = 1

def setup_paper_style():
    """Sets Matplotlib defaults for scientific publication layouts."""
    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Computer Modern", "Times New Roman", "DejaVu Serif"],
        "mathtext.fontset": "cm",
        "text.usetex": False,
        "axes.labelsize": 11.5,
        "font.size": 11,
        "legend.fontsize": 9.0,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "figure.titlesize": 12,
        "axes.titlesize": 11.5,
        "lines.linewidth": 1.4,
        "lines.markersize": 5.0,
    })

In [ ]:
from pathlib import Path
import numpy as np
import sympy as sp
DRIVE_DIRECTORY = Path("operational_res/analytical_drive_data")

def load_analytical_drive(pulse_type, drive_directory=DRIVE_DIRECTORY):
    file_path = Path(drive_directory) / f"analytical_drive_{pulse_type}.npz"

    if not file_path.exists():
        raise FileNotFoundError(f"Die Datei '{file_path}' wurde nicht gefunden.")

    with np.load(file_path, allow_pickle=False) as saved_data:
        stored_pulse_type = str(saved_data["pulse_type"].item())
        Ar_expression = sp.sympify(saved_data["Ar_srepr"].item())
        Ai_expression = sp.sympify(saved_data["Ai_srepr"].item())
        dwd_expression = sp.sympify(saved_data["dwd_srepr"].item())

        wd0_serialized = saved_data["wd0_srepr"].item()
        wd0_expression = sp.sympify(wd0_serialized) if wd0_serialized else None
    if stored_pulse_type != pulse_type:
        raise ValueError(f"Angefordert wurde '{pulse_type}', die Datei enthält jedoch '{stored_pulse_type}'.")

    pulse_data = {
        "Ar": Ar_expression,
        "Ai": Ai_expression,
        "dwd": dwd_expression,
        "wd0": wd0_expression,
    }
    print(f"Symbolische Pulsform geladen: '{file_path}'")
    return pulse_data

gauss_drive = load_analytical_drive(pulse_type="gauss")
wd0_gauss_expr = gauss_drive["wd0"]
wd0_val = float(wd0_gauss_expr)
Ar_gauss_expr = gauss_drive["Ar"].subs(wd0_gauss_expr, wd0_val)
Ai_gauss_expr = gauss_drive["Ai"].subs(wd0_gauss_expr, wd0_val)
dwd_gauss_expr = gauss_drive["dwd"].subs(wd0_gauss_expr, wd0_val)
tanh_drive = load_analytical_drive(pulse_type="tanh")
wd0_tanh_expr = tanh_drive["wd0"]
Ar_tanh_expr = tanh_drive["Ar"].subs(wd0_tanh_expr, wd0_val)
Ai_tanh_expr = tanh_drive["Ai"].subs(wd0_tanh_expr, wd0_val)
dwd_tanh_expr = tanh_drive["dwd"].subs(wd0_tanh_expr, wd0_val)


def get_symbol_by_name(expressions, symbol_name):
    free_symbols = set().union(*(expression.free_symbols for expression in expressions if expression is not None))
    matching_symbols = [symbol for symbol in free_symbols if symbol.name == symbol_name]

    if not matching_symbols:
        raise KeyError(f"Das Symbol '{symbol_name}' kommt in den geladenen Ausdrücken nicht vor.")
    return matching_symbols[0]

loaded_expressions = (Ar_gauss_expr, Ai_gauss_expr, dwd_gauss_expr)
t_sym = get_symbol_by_name(expressions=loaded_expressions, symbol_name="t")
sigma_sym = get_symbol_by_name(expressions=loaded_expressions, symbol_name=r"\sigma")
tg_sym = get_symbol_by_name(expressions=loaded_expressions, symbol_name="t_g")

display(dwd_gauss_expr)

In [ ]:
def replace_loaded_symbols(expression, t_sym, sigma_sym, tg_sym):
    symbol_map = {}

    for symbol in expression.free_symbols:
        if symbol.name == "t":
            symbol_map[symbol] = t_sym
        elif symbol.name in ("sigma", "sigma_r", r"\sigma"):
            symbol_map[symbol] = sigma_sym
        elif symbol.name in ("tg", "t_g", r"t_{g}"):
            symbol_map[symbol] = tg_sym

    return expression.xreplace(symbol_map)


def make_analytical_pulse_builder(Ar_expr, Ai_expr, dwd_expr):
    def pulse_shape_builder(t_sym, p_syms):
        tg_sym, sigma_sym, wd_offset_sym, amp_scale_sym = p_syms

        Ar = replace_loaded_symbols(expression=Ar_expr, t_sym=t_sym, sigma_sym=sigma_sym, tg_sym=tg_sym)
        Ai = replace_loaded_symbols(expression=Ai_expr, t_sym=t_sym, sigma_sym=sigma_sym, tg_sym=tg_sym)
        dwd = replace_loaded_symbols(expression=dwd_expr, t_sym=t_sym, sigma_sym=sigma_sym, tg_sym=tg_sym)

        A = amp_scale_sym * (Ar + sy.I * Ai)
        wd = wd0_val + dwd + wd_offset_sym

        return A, wd

    return pulse_shape_builder

In [ ]:
# ==============================================================================
# EINMALIGE SYMBOLISCHE FLOQUET-VORBEREITUNG
# ==============================================================================

gauss_drive = load_analytical_drive(pulse_type="gauss")

gauss_pulse_builder = make_analytical_pulse_builder(Ar_expr=gauss_drive["Ar"], Ai_expr=gauss_drive["Ai"], dwd_expr=gauss_drive["dwd"])

base_parameter_names = [
    "t_g",
    "sigma",
    "wd_offset",
    "amp_scale",
]
param_names = ["wd_offset", "amp_scale"]

def create_binder(include_geometric = False, include_micromotion = False, include_g_correction=False, verbose=False, param_names=param_names):
    H_gauss_base, M_gauss_base, M_inv_0_gauss_base = prepare_floquet_functions(pulse_shape_builder=gauss_pulse_builder, p_names=base_parameter_names, s=s, rH=rH, rW=rW, include_geometric=include_geometric, include_micromotion=include_micromotion, include_g_correction=include_g_correction, verbose=verbose)
    def bind_func(tg_target, sigma, H_base=H_gauss_base, M_base=M_gauss_base, M_inv_0_base=M_inv_0_gauss_base):
        def absolute_parameters(offsets):
            # Falls offsets als einzelnes Container-Objekt (Tupel/Liste/Array) übergeben wurde, entpacken
            if len(offsets) == 1 and isinstance(offsets[0], (tuple, list, np.ndarray)):
                offsets = offsets[0]
                
            values = dict(zip(param_names, offsets))
            return values["wd_offset"], values["amp_scale"]

        def H_func(t, *offsets):
            wd_offset, amp_scale = absolute_parameters(offsets)
            return H_base(t, tg_target, sigma, wd_offset, amp_scale)

        def M_func(t, *offsets):
            wd_offset, amp_scale = absolute_parameters(offsets)
            return M_base(t, tg_target, sigma, wd_offset, amp_scale)

        def M_inv_0_func(*offsets):
            wd_offset, amp_scale = absolute_parameters(offsets)
            return M_inv_0_base(tg_target, sigma, wd_offset, amp_scale)

        H_func.is_time_dependent = getattr(H_base, "is_time_dependent", True)
        return H_func, M_func, M_inv_0_func
    return bind_func
bind_floquet_functions = create_binder()

In [ ]:
tg_list = np.linspace(50.0, 700.0, 30)
sigma_r = 0.3
param_ranges = {"wd_offset": (-0.005, 0.05), "amp_scale": (0.8, 1.4)}
s = Simulation()

def for_tg(tg, binder, debug=False, verbose=False, low=False, plot_dynamics=False):
    def sim_pulse(tg, amp, off):
        Ar = sy.lambdify(t_sym, replace_loaded_symbols(Ar_gauss_expr, t_sym, sigma_r, tg) * amp, "numpy")
        Ai = sy.lambdify(t_sym, replace_loaded_symbols(Ai_gauss_expr, t_sym, sigma_r, tg), "numpy")
        wd = sy.lambdify(t_sym, wd0_val + replace_loaded_symbols(dwd_gauss_expr, t_sym, sigma_r, tg) + off, "numpy")

        q = lambda t, args=None: Ar(t) * np.cos(wd(t) * t) + Ai(t) * np.sin(wd(t) * t)
        tlist = np.linspace(0.0, tg, max(101, int(tg * 8)))

        columns = []
        result_a = None

        for initial_state in s.comp_indices:
            result = sesolve([s.H0, [s.V1, q]], s.E_states[initial_state], tlist)

            if initial_state == s.state_a:
                result_a = result

            columns.append([
                state.overlap(result.states[-1])
                for state in s.E_states[s.comp_indices]
            ])

        U = np.column_stack(columns)
        return infid_population(U), q, tlist, result_a

    H_f, M_f, M_i0_f = binder(tg, sigma_r)
    popsize, maxiter, tol = (8, 40, 1e-5) if low else (17, 80, 1e-8)

    p_opt, f_inf = optimize_pulse_parameters(
        H_f, M_f, M_i0_f, param_ranges, s, tg,
        popsize=popsize, maxiter=maxiter, tol=tol, verbose=verbose
    )

    infid_direct, _, tlist, result_a = sim_pulse(
        tg, p_opt["amp_scale"], p_opt["wd_offset"]
    )

    if plot_dynamics:
        states = [s.state_a, s.state_b]
        pops_direct = np.array([
            [abs(s.E_states[state].overlap(psi))**2 for psi in result_a.states]
            for state in states
        ])

        # Floquet-Dynamik aus |a>
        p_vals = (p_opt["wd_offset"], p_opt["amp_scale"])
        t_mids = 0.5 * (tlist[:-1] + tlist[1:])
        dt = tlist[1] - tlist[0]

        if hasattr(H_f, "evaluate_stack"):
            H_all = H_f.evaluate_stack(t_mids, *p_vals)
        elif getattr(H_f, "is_time_dependent", True):
            H_all = np.stack([
                np.asarray(H_f(t, *p_vals), dtype=complex).reshape(s.d_res, s.d_res)
                for t in t_mids
            ], axis=2)
        else:
            H_static = np.asarray(
                H_f(t_mids[0], *p_vals), dtype=complex
            ).reshape(s.d_res, s.d_res)

            H_all = np.repeat(
                H_static[:, :, np.newaxis], len(t_mids), axis=2
            )

        M_inv_0 = np.asarray(
            M_i0_f(*p_vals), dtype=complex
        ).reshape(s.d_res, s.D)

        phi_a = M_inv_0[:, s.state_a]
        U_eff = np.eye(s.d_res, dtype=complex)
        pops_floquet = np.zeros((2, len(tlist)))

        for k, time in enumerate(tlist):
            M = np.asarray(
                M_f(time, *p_vals), dtype=complex
            ).reshape(s.D, s.d_res)

            psi_floquet = M @ U_eff @ phi_a
            norm = np.linalg.norm(psi_floquet)

            if norm < 1e-14:
                raise ValueError(
                    f"Floquet state has vanishing norm at t={time}."
                )

            psi_floquet /= norm
            pops_floquet[:, k] = np.abs(
                psi_floquet[[s.state_a, s.state_b]]
            )**2

            if k < len(t_mids):
                U_eff = scipy_linalg.expm(
                    -1j * H_all[:, :, k] * dt
                ) @ U_eff

        # Plot
        x = tlist / tg
        fig, ax = plt.subplots(figsize=(6.2, 3.2), constrained_layout=True)

        ax.plot(
            x, pops_direct[0],
            color="#1f77b4", lw=1.6,
            label=r"Direct $|\langle a|\psi(t)\rangle|^2$"
        )
        ax.plot(
            x, pops_floquet[0],
            color="black", ls="--", lw=1.8, zorder=5,
            label=r"FPT $|\langle a|\psi(t)\rangle|^2$"
        )
        ax.plot(
            x, pops_direct[1],
            color="#ff7f0e", lw=1.6,
            label=r"Direct $|\langle b|\psi(t)\rangle|^2$"
        )
        ax.plot(
            x, pops_floquet[1],
            color="black", ls="-.", lw=1.8, zorder=5,
            label=r"FPT $|\langle b|\psi(t)\rangle|^2$"
        )

        ax.set_title(r"Initial state $\vert\psi(0)\rangle=\vert a\rangle$")
        ax.set_xlabel(r"$t/t_{\mathrm{g}}$")
        ax.set_ylabel(r"$|\langle i|\psi(t)\rangle|^2$")
        ax.set_xlim(0.0, 1.0)
        ax.set_ylim(-0.02, 1.02)
        ax.grid(ls=":", lw=0.7, alpha=0.4)
        ax.tick_params(direction="in")
        ax.legend(frameon=False, ncol=2)

        plt.show()

    if debug:
        print(
            "infid_direct:", infid_direct,
            "f_inf:", f_inf,
            "p_opt:", p_opt
        )
    return infid_direct, f_inf, p_opt


def run_tg_sweep(save_name, binder, debug=False, verbose=False, low=False):
    linfid_pop_direct, linfid_pop_floq, linfid_unopt_direct, linfid_unopt_floq = [], [], [], []
    for tg in tqdm(tg_list):
        infid_direct, f_inf, _ = for_tg(tg, binder, debug=debug, verbose=verbose, low=low)
        linfid_pop_direct.append(infid_direct)
        linfid_pop_floq.append(f_inf)        
    Path("operational_res/drag_popfids").mkdir(parents=True, exist_ok=True)
    np.savez(f"operational_res/drag_popfids/drag_popfids_data_{save_name}.npz", tg_list=tg_list, direct=linfid_pop_direct, floq=linfid_pop_floq, unopt_direct=linfid_unopt_direct, unopt_floq=linfid_unopt_floq)

Determine optimal parameters for $t_g=200$~ns

In [ ]:
inf_direct, inf_floquet, p_opt = for_tg(200, bind_floquet_functions, low=True, plot_dynamics=True)
print("floquet: ", inf_floquet, " direct: ", inf_direct, " p_opt: ", p_opt)

In [ ]:
run_tg_sweep("standard", bind_floquet_functions)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

def load_and_plot(save_name):
    # Daten laden
    ref_data = np.load("operational_res/cos_popfids_data.npz")
    ref_tg_list, ref_infid_pop = ref_data["tg_list"], ref_data["infid_pop_list"]

    drag_data = np.load(f"operational_res/drag_popfids/drag_popfids_data_{save_name}.npz")
    tg_list = drag_data["tg_list"]
    linfid_pop_direct = drag_data["direct"]
    linfid_pop_floq = drag_data["floq"]
    def plot_infidelity_vs_tg_fullwidth(
        tg_list, 
        linfid_pop_direct, 
        linfid_pop_floq, 
        ref_tg_list=None, 
        ref_infid_pop=None, 
        title_suffix="",
        save_path=None, 
        show=True
    ):
        setup_paper_style()
        
        # Farben
        c_direct, c_floq, c_ref = "#1f77b4", "#d62728", "#555555"
        c_unopt_dir, c_unopt_flq = "#17becf", "#ff7f0e"  # Cyan & Orange für unoptimiert

        fullwidth_style = {
            "axes.labelsize": 12.0, "axes.titlesize": 12.5,
            "xtick.labelsize": 10.5, "ytick.labelsize": 10.5,
            "legend.fontsize": 9.5, "lines.linewidth": 1.8, "lines.markersize": 5.5,
        }
        
        with plt.rc_context(fullwidth_style):
            fig, ax = plt.subplots(figsize=(6.8, 3.8), dpi=300)
            
            # 1. Cosine Reference
            if ref_tg_list is not None and ref_infid_pop is not None:
                ax.semilogy(
                    ref_tg_list, ref_infid_pop, ":", color=c_ref, marker="^", 
                    mfc="white", mec=c_ref, mew=1.2, lw=1.5, label=r"Cosine Reference ($I_{\mathrm{pop}}$)"
                )
            # 3. Optimierte Kurven
            ax.semilogy(
                tg_list, linfid_pop_direct, "-", color=c_direct, marker="o", 
                mfc="white", mec=c_direct, mew=1.5, label=r"DRAG Direct ($U_{\mathrm{direct}}$)"
            )
            ax.semilogy(
                tg_list, linfid_pop_floq, "--", color=c_floq, marker="s", 
                mfc="white", mec=c_floq, mew=1.5, label=r"DRAG Floquet ($U_{\mathrm{floq}}$)"
            )
            
            ax.set_xlabel(r"Gate duration $t_g$ [ns]")
            ax.set_ylabel(r"Population Infidelity $I_{\mathrm{pop}}$")
            ax.set_title(rf"Population Infidelity vs. Gate Duration $t_g${title_suffix}", fontweight="bold")
            ax.grid(True, which="both", linestyle=":", alpha=0.5, linewidth=0.8)
            ax.legend(loc="best", frameon=True, fancybox=False, edgecolor="gray", framealpha=0.9)
            
            fig.tight_layout()
            if save_path:
                out = Path(save_path)
                out.parent.mkdir(parents=True, exist_ok=True)
                fig.savefig(out, bbox_inches="tight")
            if show:
                plt.show()
                
            return fig, ax

    # Plot 1: Standard (nur optimierte DRAG-Daten)
    fig1, ax1 = plot_infidelity_vs_tg_fullwidth(
        tg_list, linfid_pop_direct, linfid_pop_floq,
        ref_tg_list=ref_tg_list, ref_infid_pop=ref_infid_pop,
        save_path=f"Figure/drag_results/{save_name}.pdf"
    )
# load_and_plot("standard")

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

def load_and_plot_standard_custom(save_path="Figure/drag_results/standard_custom_comparison.pdf", show=True, use_FAPT=False):
    save_path = "Figure/drag_results/standard_custom_comparison.pdf" if not use_FAPT else "Figure/drag_results/standard_custom_comparison_FAPT.pdf"
    # Daten laden
    name_addition = "" if not use_FAPT else "_FAPT"
    cos_data = np.load(f"operational_res/cos_popfids_data.npz")
    standard_data = np.load(f"operational_res/drag_popfids/drag_popfids_data_standard{name_addition}.npz")
    custom_data = np.load(f"operational_res/drag_popfids/drag_popfids_data_custom{name_addition}.npz")

    tg_cos, infid_cos = cos_data["tg_list"], cos_data["infid_pop_list"]
    tg_standard, direct_standard, floq_standard = standard_data["tg_list"], standard_data["direct"], standard_data["floq"]
    tg_custom, direct_custom, floq_custom = custom_data["tg_list"], custom_data["direct"], custom_data["floq"]

    setup_paper_style()

    c_cos = "#1f77b4"
    c_standard = "#d62728"
    c_custom = "#2ca02c"

    plot_style = {
        "axes.labelsize": 12.0,
        "axes.titlesize": 12.5,
        "xtick.labelsize": 10.5,
        "ytick.labelsize": 10.5,
        "legend.fontsize": 9.0,
        "lines.linewidth": 2.0,
        "lines.markersize": 6.0,
    }

    with plt.rc_context(plot_style):
        fig, ax = plt.subplots(figsize=(6.8, 3.8), dpi=300)

        # Cosine reference: gleicher Stil wie im bisherigen Cosine-Plot
        ax.semilogy(tg_cos, infid_cos, "-", color=c_cos, marker=".", markerfacecolor="white", markeredgecolor=c_cos, markeredgewidth=1.8, label=r"Cosine (Direct)")

        # Standard DRAG
        ax.semilogy(tg_standard, direct_standard, "-", color=c_standard, marker="o", mfc="white", mec=c_standard, mew=1.5, label=r"Standard DRAG (Direct)")
        ax.semilogy(tg_standard, floq_standard, "--", color=c_standard, marker="o", mfc="white", mec=c_standard, mew=1.0, lw=1.4, ms=4.5, alpha=0.40, label=r"Standard DRAG (Floquet)")

        # Custom DRAG
        ax.semilogy(tg_custom, direct_custom, "-", color=c_custom, marker="s", mfc="white", mec=c_custom, mew=1.5, label=r"Custom DRAG (Direct)")
        ax.semilogy(tg_custom, floq_custom, "--", color=c_custom, marker="s", mfc="white", mec=c_custom, mew=1.0, lw=1.4, ms=4.5, alpha=0.40, label=r"Custom DRAG (Floquet)")

        ax.set_xlabel(r"Gate duration $t_g$ [ns]")
        ax.set_ylabel(r"Infidelity $I_{\mathrm{pop}}$")
        ax.set_title(r"Population infidelity", fontweight="bold")
        ax.grid(True, which="both", linestyle=":", alpha=0.5, linewidth=0.8)
        ax.legend(loc="best", ncol=2, frameon=True, fancybox=False, edgecolor="gray", framealpha=0.9)

        fig.tight_layout()

        if save_path:
            output_file = Path(save_path)
            output_file.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(output_file, bbox_inches="tight")

        if show:
            plt.show()

        return fig, ax

fig, ax = load_and_plot_standard_custom()

In [ ]:
binder = create_binder(include_geometric=False, include_g_correction=False, include_micromotion=True, verbose=False)

In [ ]:
run_tg_sweep("standard_FAPT", binder = binder, debug=True, verbose=False, low=False)
# True True True 1.7996611635373583e-05
# True True False 1.8021649942356888e-05
# True False False 1.800959878617281e-05

In [14]:
load_and_plot_standard_custom(use_FAPT=True)

FileNotFoundError: [Errno 2] No such file or directory: 'operational_res/cos_popfids_data_FAPT.npz'